# P04 · Rendimiento y análisis reproducible con datos PQRS completos

**Cobertura:** seis cortes semestrales completos de 2023, 2024 y 2025.  
**Motores:** Polars y DuckDB para comprobar la lectura; Polars y pandas para comparar rendimiento.

## Propósito

Construir primero un corpus confiable y solo después medir rendimiento. El objetivo no es declarar un ganador universal, sino explicar qué ocurre con **estos datos, esta consulta, este equipo y estas versiones**.

Al finalizar podremos responder:

1. ¿Tenemos realmente los seis cortes oficiales y podemos reconstruir su procedencia?
2. ¿Polars y DuckDB leen la misma población y producen el mismo agregado?
3. ¿Qué diferencias de tiempo y memoria aparecen entre Polars y pandas cuando resuelven exactamente el mismo trabajo?


## Por qué importa y dónde se aplica

Una comparación de motores carece de valor si cada uno procesa filas, columnas o reglas distintas. En ingeniería de datos este protocolo se aplica al seleccionar una tecnología, justificar costos de infraestructura, migrar procesos y definir pruebas de regresión.

```text
Portal oficial
      │
      ▼
Manifiesto + descarga + SHA-256
      │
      ▼
6 CSV ──► lectura Polars ─┐
      └─► lectura DuckDB ─┴─► equivalencia semántica
                                  │
                                  ▼
                          Parquet común
                         ┌────────┴────────┐
                         ▼                 ▼
                      Polars            pandas
                         └────────┬────────┘
                                  ▼
                    tiempo + memoria + límites
```

La descarga, la normalización y la creación del Parquet son preparación. **No forman parte del tiempo del benchmark.**


## Criterios de éxito

- Están presentes los seis cortes y suman **5.327.501 registros**.
- El inventario conserva URL, tamaño, huella y estado de cada fuente.
- Polars y DuckDB coinciden por corte y en el agregado de control.
- Polars y pandas leen el mismo Parquet, proyectan las mismas columnas y producen la misma salida.
- Cada motor tiene una vuelta de calentamiento y tres repeticiones medidas en procesos nuevos.
- La conclusión registra versiones, configuración, tiempo, memoria y límites del experimento.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import duckdb
import pandas as pd
import polars as pl
import psutil
import pyarrow as pa
from IPython.display import display

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('Polars:', pl.__version__)
print('DuckDB:', duckdb.__version__)
print('PyArrow:', pa.__version__)


In [ ]:
# Funciona si Jupyter abre el notebook desde kit/ o desde kit/Notebooks/.
CWD = Path.cwd().resolve()
if (CWD / '00_datos.py').exists():
    KIT = CWD
elif (CWD.parent / '00_datos.py').exists():
    KIT = CWD.parent
else:
    raise RuntimeError('Abra este notebook desde kit/ o kit/Notebooks/.')

DATOS = KIT / 'data' / 'pqrs_completos'
SALIDA = KIT / 'salidas' / 'pqrs_completos_2023_2025'
PARQUET_COMUN = SALIDA / 'pqrs_2023_2025_comun.parquet'
SALIDA.mkdir(parents=True, exist_ok=True)

print('KIT:', KIT)
print('Datos:', DATOS)
print('Salidas:', SALIDA)


## Fase 1 · Descargar y verificar las fuentes

Las fuentes son datos abiertos de la Superintendencia Nacional de Salud. Los cortes 2025 se descargan directamente desde:

- 2025-I: <https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/289c737231d14a968224150c585d08a3/data>
- 2025-II: <https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/d36192e86d874296aeaf95cd1a53e904/data>

La segunda URL entrega un ZIP. El kit conserva el original y extrae `Reclamos_Salud_Semestre_2_2025.txt` como `PQRD_2025_II.csv`. Los motores analíticos leen el CSV extraído, nunca el ZIP.

### Proceso por terminal

```bash
cd kit
python 00_datos.py --listar
python 00_datos.py --descargar pqrs-completos
python 00_datos.py --verificar pqrs-completos
```

La descarga ocupa aproximadamente 3,830 GB descomprimidos. No abra 2025-II en Excel: supera su límite de filas.


In [ ]:
# Cambie a True únicamente cuando quiera iniciar una descarga de varios GB.
DESCARGAR = False

subprocess.run([sys.executable, '00_datos.py', '--listar'], cwd=KIT, check=True)
if DESCARGAR:
    subprocess.run(
        [sys.executable, '00_datos.py', '--descargar', 'pqrs-completos'],
        cwd=KIT, check=True
    )
else:
    print('Descarga omitida. Cambie DESCARGAR=True o ejecute el comando en la terminal.')


In [ ]:
# Esta verificación es obligatoria y falla si falta un corte o cambia su contenido.
subprocess.run(
    [sys.executable, '00_datos.py', '--verificar', 'pqrs-completos'],
    cwd=KIT, check=True
)


### Inventario reproducible

El manifiesto expresa lo esperado; el inventario registra lo observado localmente. Una huella SHA-256 permite detectar archivos truncados o sustituidos aunque conserven el mismo nombre. Calcular seis huellas sobre 3,830 GB toma tiempo, pero ocurre fuera del benchmark.


In [ ]:
def cargar_fuentes():
    base = json.loads((KIT / 'pqrs_fuentes.json').read_text(encoding='utf-8'))
    extra = json.loads((KIT / 'pqrs_fuentes_adicionales_2023_2025.json').read_text(encoding='utf-8'))
    return sorted(base + extra, key=lambda f: f['periodo'])

def sha256_archivo(ruta, bloque=8 * 1024 * 1024):
    huella = hashlib.sha256()
    with ruta.open('rb') as archivo:
        for fragmento in iter(lambda: archivo.read(bloque), b''):
            huella.update(fragmento)
    return huella.hexdigest()

fuentes = cargar_fuentes()
pd.DataFrame([{
    'corte': f['periodo'], 'archivo': f['archivo'], 'filas_esperadas': f['filas'],
    'columnas_origen': len(f['columnas']), 'separador': f['separador'],
    'contenedor': f.get('contenedor', 'csv'), 'url': f['url']
} for f in fuentes])


In [ ]:
CALCULAR_SHA256 = True
registros_inventario = []
for fuente in fuentes:
    ruta = DATOS / fuente['archivo']
    existe = ruta.exists()
    huella_local = sha256_archivo(ruta) if existe and CALCULAR_SHA256 else None
    coincide = (
        existe
        and ruta.stat().st_size == fuente['bytes']
        and (not CALCULAR_SHA256 or huella_local == fuente['sha256'])
    )
    registros_inventario.append({
        'corte': fuente['periodo'],
        'archivo': fuente['archivo'],
        'url': fuente['url'],
        'contenedor_origen': fuente.get('contenedor', 'csv'),
        'separador': fuente['separador'],
        'codificacion': fuente.get('codificacion', 'utf-8-sig'),
        'bytes_esperados': fuente['bytes'],
        'bytes_locales': ruta.stat().st_size if existe else None,
        'sha256_esperado': fuente['sha256'],
        'sha256_local': huella_local,
        'fecha_revision_utc': datetime.now(timezone.utc).isoformat(),
        'estado': 'coincide' if coincide else ('sin_huella' if existe else 'ausente')
    })

inventario = pd.DataFrame(registros_inventario)
inventario.to_csv(SALIDA / 'inventario_fuentes.csv', index=False)
display(inventario)
assert inventario['estado'].eq('coincide').all(), 'Deténgase: una o más fuentes no coinciden.'


## Fase 2 · Lectura inicial Polars–DuckDB

Los esquemas de origen no son idénticos. En 2023-I, por ejemplo, `AÑO`, `MES` y `PQR_CANAL` están en mayúsculas; 2024-I usa coma y los demás cortes usan punto y coma. No se corrigen todos los campos: se proyecta el mínimo común necesario.

La proyección común es:

```text
corte_archivo, periodo, mes, pqr_canal, afec_cod_depto, afec_cod_mpio
```

Esta fase no compara velocidad. Su función es detectar diferencias de población, separador, nombres, tipos y nulos antes de medir.


In [ ]:
CANDIDATOS = {
    'periodo': ['periodo', 'AÑO'],
    'mes': ['mes', 'MES'],
    'pqr_canal': ['pqr_canal', 'PQR_CANAL'],
    'afec_cod_depto': ['afec_cod_depto', 'AFEC_COD_DEPTO'],
    'afec_cod_mpio': ['afec_cod_mpio', 'AFEC_COD_MPIO'],
}
COLUMNAS_COMUNES = ['corte_archivo'] + list(CANDIDATOS)

def columna_origen(fuente, canonica):
    indice = {nombre.casefold(): nombre for nombre in fuente['columnas']}
    for candidata in CANDIDATOS[canonica]:
        if candidata.casefold() in indice:
            return indice[candidata.casefold()]
    raise KeyError(f"{fuente['periodo']}: falta {canonica}")

def texto_limpio(nombre):
    return pl.col(nombre).cast(pl.String).str.strip_chars().replace('', None)

lazy_por_corte = []
for fuente in fuentes:
    ruta = DATOS / fuente['archivo']
    nombres = {c: columna_origen(fuente, c) for c in CANDIDATOS}
    lectura = pl.scan_csv(
        ruta, separator=fuente['separador'], encoding='utf8-lossy',
        infer_schema_length=0,
        schema_overrides={nombre: pl.String for nombre in fuente['columnas']},
        ignore_errors=False
    )
    lazy_por_corte.append(lectura.select(
        pl.lit(fuente['periodo']).alias('corte_archivo'),
        texto_limpio(nombres['periodo']).cast(pl.Int16, strict=False).alias('periodo'),
        texto_limpio(nombres['mes']).cast(pl.Int8, strict=False).alias('mes'),
        texto_limpio(nombres['pqr_canal']).alias('pqr_canal'),
        texto_limpio(nombres['afec_cod_depto']).alias('afec_cod_depto'),
        texto_limpio(nombres['afec_cod_mpio']).alias('afec_cod_mpio'),
    ))

pqrs_polars = pl.concat(lazy_por_corte, how='vertical')
print(pqrs_polars.explain(optimized=True))


In [ ]:
def sql_identificador(nombre):
    return '"' + nombre.replace('"', '""') + '"'

def sql_texto(nombre):
    campo = sql_identificador(nombre)
    return f"NULLIF(TRIM(CAST({campo} AS VARCHAR)), '')"

consultas = []
for fuente in fuentes:
    ruta_sql = str(DATOS / fuente['archivo']).replace("'", "''")
    corte_sql = fuente['periodo'].replace("'", "''")
    separador_sql = fuente['separador'].replace("'", "''")
    nombres = {c: columna_origen(fuente, c) for c in CANDIDATOS}
    consultas.append(f"""
        SELECT
            '{corte_sql}' AS corte_archivo,
            TRY_CAST({sql_identificador(nombres['periodo'])} AS SMALLINT) AS periodo,
            TRY_CAST({sql_identificador(nombres['mes'])} AS TINYINT) AS mes,
            {sql_texto(nombres['pqr_canal'])} AS pqr_canal,
            {sql_texto(nombres['afec_cod_depto'])} AS afec_cod_depto,
            {sql_texto(nombres['afec_cod_mpio'])} AS afec_cod_mpio
        FROM read_csv('{ruta_sql}', delim='{separador_sql}', header=true,
                      all_varchar=true, strict_mode=true)
    """)

conexion = duckdb.connect()
conexion.execute('CREATE OR REPLACE TEMP VIEW pqrs_duckdb AS ' + ' UNION ALL '.join(consultas))
print(conexion.execute('DESCRIBE pqrs_duckdb').df())


In [ ]:
control_polars = (
    pqrs_polars.group_by('corte_archivo').agg(
        pl.len().alias('filas_polars'),
        pl.col('periodo').min().alias('periodo_min_polars'),
        pl.col('periodo').max().alias('periodo_max_polars'),
        pl.col('mes').min().alias('mes_min_polars'),
        pl.col('mes').max().alias('mes_max_polars'),
        pl.col('afec_cod_depto').is_null().sum().alias('depto_nulo_polars'),
        pl.col('afec_cod_mpio').is_null().sum().alias('mpio_nulo_polars'),
    ).sort('corte_archivo').collect(engine='streaming').to_pandas()
)

control_duckdb = conexion.execute('''
    SELECT corte_archivo, COUNT(*) AS filas_duckdb,
           MIN(periodo) AS periodo_min_duckdb, MAX(periodo) AS periodo_max_duckdb,
           MIN(mes) AS mes_min_duckdb, MAX(mes) AS mes_max_duckdb,
           COUNT(*) FILTER (WHERE afec_cod_depto IS NULL) AS depto_nulo_duckdb,
           COUNT(*) FILTER (WHERE afec_cod_mpio IS NULL) AS mpio_nulo_duckdb
    FROM pqrs_duckdb GROUP BY corte_archivo ORDER BY corte_archivo
''').df()

esperado = pd.DataFrame([{
    'corte_archivo': f['periodo'], 'filas_esperadas': f['filas'],
    'columnas_origen': len(f['columnas'])
} for f in fuentes])
control = esperado.merge(control_polars, on='corte_archivo').merge(control_duckdb, on='corte_archivo')
control.to_csv(SALIDA / 'control_lectura_polars_duckdb.csv', index=False)
display(control)
assert (control['filas_esperadas'] == control['filas_polars']).all()
assert (control['filas_polars'] == control['filas_duckdb']).all()


In [ ]:
CLAVES_CONTROL = ['corte_archivo', 'periodo', 'mes', 'afec_cod_depto']
agregado_polars = (
    pqrs_polars.group_by(CLAVES_CONTROL).agg(pl.len().alias('reportes'))
    .collect(engine='streaming').to_pandas()
)
agregado_duckdb = conexion.execute('''
    SELECT corte_archivo, periodo, mes, afec_cod_depto, COUNT(*) AS reportes
    FROM pqrs_duckdb
    GROUP BY corte_archivo, periodo, mes, afec_cod_depto
''').df()

def normalizar_agregado(df):
    salida = df.copy()
    salida['afec_cod_depto'] = salida['afec_cod_depto'].fillna('<NULL>').astype(str)
    return salida.sort_values(CLAVES_CONTROL, na_position='first').reset_index(drop=True)

polars_normalizado = normalizar_agregado(agregado_polars)
duckdb_normalizado = normalizar_agregado(agregado_duckdb)
pd.testing.assert_frame_equal(polars_normalizado, duckdb_normalizado, check_dtype=False)

equivalencia_inicial = {
    'equivalentes': True,
    'filas_totales': int(control['filas_polars'].sum()),
    'grupos_comparados': len(polars_normalizado),
    'claves': CLAVES_CONTROL,
}
(SALIDA / 'equivalencia_polars_duckdb.json').write_text(
    json.dumps(equivalencia_inicial, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([equivalencia_inicial]))


### Ruptura semántica que debe documentarse

La clasificación de motivos cambió el 1 de julio de 2023. Por eso este laboratorio no interpreta tendencias de motivos entre 2023-I y los cortes posteriores. La comparación se limita a columnas comunes cuya semántica puede justificarse.

**Regla:** si la equivalencia anterior falla, no continúe al benchmark. Primero explique y resuelva la diferencia.


## Fase 3 · Construir una entrada común

Parquet conserva tipos y permite proyectar columnas sin recorrer información innecesaria. Usar un solo archivo común elimina una fuente importante de sesgo: ambos motores parten de los mismos bytes y del mismo esquema lógico.


In [ ]:
pqrs_polars.sink_parquet(PARQUET_COMUN, compression='zstd')

total_esperado = sum(f['filas'] for f in fuentes)
control_parquet = (
    pl.scan_parquet(PARQUET_COMUN)
    .group_by('corte_archivo').agg(pl.len().alias('filas'))
    .sort('corte_archivo').collect().to_pandas()
)
assert int(control_parquet['filas'].sum()) == total_esperado == 5_327_501
assert set(control_parquet['corte_archivo']) == {f['periodo'] for f in fuentes}
display(control_parquet)
print(f'Parquet común: {PARQUET_COMUN} ({PARQUET_COMUN.stat().st_size / 1e9:.3f} GB)')


## Fase 4 · Benchmark Polars–pandas

### Trabajo equivalente

1. Leer cuatro columnas del Parquet común.
2. Agrupar por corte, año, mes y departamento afectado.
3. Contar reportes, incluyendo departamentos nulos.
4. Materializar el resultado.

### Frontera del reloj

Incluye lectura Parquet, proyección, agrupación, conteo y materialización. Excluye ordenación para comparar, escritura de resultados, controles y gráficos. Cada ejecución ocurre en un proceso nuevo; el proceso padre mide el pico RSS con el mismo intervalo de sondeo.

Se realiza un calentamiento no registrado por motor y luego tres repeticiones alternando el orden. La alternancia reduce, pero no elimina, el efecto de caché.


In [ ]:
WORKER = r'''
import json
import sys
import time

motor, entrada, salida = sys.argv[1:4]
claves = ['corte_archivo', 'periodo', 'mes', 'afec_cod_depto']

if motor == 'pandas':
    import pandas as pd
    inicio = time.perf_counter()
    datos = pd.read_parquet(entrada, columns=claves)
    resultado = (datos.groupby(claves, dropna=False, sort=False)
                .size().rename('reportes').reset_index())
    segundos = time.perf_counter() - inicio
    filas_entrada = len(datos)
    total = int(resultado['reportes'].sum())
    resultado = resultado.sort_values(claves, na_position='first').reset_index(drop=True)
    resultado.to_parquet(salida, index=False)
elif motor == 'polars':
    import polars as pl
    inicio = time.perf_counter()
    resultado = (pl.scan_parquet(entrada).select(claves)
                 .group_by(claves).agg(pl.len().alias('reportes'))
                 .collect(engine='streaming'))
    segundos = time.perf_counter() - inicio
    filas_entrada = int(resultado['reportes'].sum())
    total = filas_entrada
    resultado.sort(claves, nulls_last=False).write_parquet(salida)
else:
    raise ValueError(motor)

print(json.dumps({'motor': motor, 'segundos': segundos,
                  'filas_entrada': filas_entrada, 'total_resultado': total,
                  'grupos': len(resultado)}))
'''

def rss_arbol(proceso):
    try:
        raiz = psutil.Process(proceso.pid)
        procesos = [raiz] + raiz.children(recursive=True)
        return sum(p.memory_info().rss for p in procesos if p.is_running())
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return 0

def ejecutar_motor(motor, etiqueta, intervalo=0.01):
    archivo_resultado = SALIDA / f'resultado_{etiqueta}.parquet'
    proceso = subprocess.Popen(
        [sys.executable, '-c', WORKER, motor, str(PARQUET_COMUN), str(archivo_resultado)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    pico_rss = 0
    while proceso.poll() is None:
        pico_rss = max(pico_rss, rss_arbol(proceso))
        time.sleep(intervalo)
    stdout, stderr = proceso.communicate()
    if proceso.returncode != 0:
        raise RuntimeError(f'{motor} falló: {stderr}')
    medicion = json.loads(stdout.strip().splitlines()[-1])
    medicion.update({
        'etiqueta': etiqueta, 'pico_rss_bytes': pico_rss,
        'intervalo_sondeo_s': intervalo, 'archivo_resultado': str(archivo_resultado)
    })
    return medicion


In [ ]:
# Calentamiento: se ejecuta, pero no se incorpora al resumen.
calentamientos = [
    ejecutar_motor('pandas', 'calentamiento_pandas'),
    ejecutar_motor('polars', 'calentamiento_polars'),
]
display(pd.DataFrame(calentamientos))

orden = [
    (1, 'pandas'), (1, 'polars'),
    (2, 'polars'), (2, 'pandas'),
    (3, 'pandas'), (3, 'polars'),
]
mediciones = []
for repeticion, motor in orden:
    medicion = ejecutar_motor(motor, f'r{repeticion}_{motor}')
    medicion['repeticion'] = repeticion
    mediciones.append(medicion)
    print(motor, repeticion, f"{medicion['segundos']:.3f} s",
          f"{medicion['pico_rss_bytes'] / 1e6:.1f} MB RSS")

benchmark = pd.DataFrame(mediciones)
benchmark.to_csv(SALIDA / 'benchmark_polars_pandas.csv', index=False)
display(benchmark)


In [ ]:
resultado_pandas = pd.read_parquet(SALIDA / 'resultado_r1_pandas.parquet')
resultado_polars = pd.read_parquet(SALIDA / 'resultado_r1_polars.parquet')
resultado_pandas = normalizar_agregado(resultado_pandas)
resultado_polars = normalizar_agregado(resultado_polars)
pd.testing.assert_frame_equal(resultado_pandas, resultado_polars, check_dtype=False)
assert int(resultado_pandas['reportes'].sum()) == 5_327_501

control_benchmark = {
    'equivalentes': True,
    'filas_entrada': 5_327_501,
    'grupos_comparados': len(resultado_pandas),
    'consulta': 'conteo por corte_archivo, periodo, mes y afec_cod_depto',
}
(SALIDA / 'equivalencia_polars_pandas.json').write_text(
    json.dumps(control_benchmark, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([control_benchmark]))


In [ ]:
resumen = (benchmark.groupby('motor', as_index=False)
    .agg(segundos_mediana=('segundos', 'median'),
         segundos_min=('segundos', 'min'),
         segundos_max=('segundos', 'max'),
         segundos_desviacion=('segundos', 'std'),
         rss_mediana_bytes=('pico_rss_bytes', 'median'),
         rss_min_bytes=('pico_rss_bytes', 'min'),
         rss_max_bytes=('pico_rss_bytes', 'max')))
resumen.to_csv(SALIDA / 'benchmark_polars_pandas_resumen.csv', index=False)

disco = shutil.disk_usage(PARQUET_COMUN.parent)
configuracion = {
    'fecha_utc': datetime.now(timezone.utc).isoformat(),
    'sistema': platform.platform(),
    'procesador': platform.processor(),
    'cpu_logicas': psutil.cpu_count(logical=True),
    'cpu_fisicas': psutil.cpu_count(logical=False),
    'ram_bytes': psutil.virtual_memory().total,
    'disco_libre_bytes': disco.free,
    'python': platform.python_version(),
    'pandas': pd.__version__, 'polars': pl.__version__,
    'duckdb': duckdb.__version__, 'pyarrow': pa.__version__,
    'psutil': psutil.__version__,
    'hilos_polars': pl.thread_pool_size(),
    'POLARS_MAX_THREADS': os.environ.get('POLARS_MAX_THREADS'),
    'intervalo_sondeo_rss_s': 0.01,
    'frontera': 'lectura Parquet + proyección + groupby + conteo + materialización',
}
(SALIDA / 'configuracion_benchmark.json').write_text(
    json.dumps(configuracion, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(resumen)
display(pd.DataFrame([configuracion]))


## Interpretación guiada

Redacte la conclusión en este orden:

1. **Equivalencia:** confirme las filas y grupos comparados. Sin equivalencia no hay comparación válida.
2. **Resultado observado:** compare medianas, rango y pico RSS; no escoja solo el mejor tiempo.
3. **Mecanismos:** relacione el resultado con lectura columnar, proyección, ejecución lazy, paralelismo, materialización y presión de memoria.
4. **Factores externos:** explique caché del sistema operativo, almacenamiento, procesos concurrentes e intervalo de sondeo.
5. **Alcance:** limite la afirmación a esta consulta, este conjunto, este equipo y estas versiones.

Preguntas para discusión:

- ¿Cambiaría el resultado con más columnas o con una unión grande?
- ¿La diferencia proviene solo del motor o también de sus estrategias internas?
- ¿Qué costo de desarrollo, operación y compatibilidad acompañaría una migración?
- ¿Qué nueva prueba necesitaría antes de recomendar un motor para producción?

No concluya que un motor “siempre es más rápido”.


## Productos de entrega

La carpeta `kit/salidas/pqrs_completos_2023_2025/` debe contener:

- `inventario_fuentes.csv`;
- `control_lectura_polars_duckdb.csv`;
- `equivalencia_polars_duckdb.json`;
- el Parquet común;
- `benchmark_polars_pandas.csv`;
- `benchmark_polars_pandas_resumen.csv`;
- `equivalencia_polars_pandas.json`;
- `configuracion_benchmark.json`;
- notebook ejecutado e informe breve con interpretación, límites y decisión técnica.

Los CSV originales no se adjuntan, pero su procedencia, tamaño y huella deben permitir reconstruir el ejercicio.
